In [8]:
import torch
from torch import nn
from d2l import torch as d2l

In [4]:
# Multiple Input Channel Cross-Correlation


def corr2d_multi_in(
    X: torch.Tensor,  # (in, H, W): (2, 3, 3)
    K: torch.Tensor,  # (in, H, W): (2, 2, 2) / out은 외부 loop에서
) -> torch.Tensor:

    # 각 Channel 끼리 Convolution 연산: [2, 2, 2]
    channel_outputs: list[torch.Tensor] = [
        d2l.corr2d(
            X_channel,
            K_channel,
        )
        for X_channel, K_channel in zip(X, K)
    ]

    # 커널 통과 후 합치기
    return torch.stack(
        channel_outputs,  # [2, 2, 2]
        dim=0,
    ).sum(
        dim=0
    )  # [2, 2]

In [5]:
# Two-Channel Input 검증

# X: [2, 3, 3]
X = torch.tensor(
    [
        [
            [0.0, 1.0, 2.0],
            [3.0, 4.0, 5.0],
            [6.0, 7.0, 8.0],
        ],
        [
            [1.0, 2.0, 3.0],
            [4.0, 5.0, 6.0],
            [7.0, 8.0, 9.0],
        ],
    ]
)

# K: [2, 2, 2]
K = torch.tensor(
    [
        [
            [0.0, 1.0],
            [2.0, 3.0],
        ],
        [
            [1.0, 2.0],
            [3.0, 4.0],
        ],
    ]
)

# [2, 2]
Y = corr2d_multi_in(
    X,  # [2, 3, 3]
    K,  # [2, 2, 2]
)

print("X shape:", X.shape)
print("K shape:", K.shape)
print("Y shape:", Y.shape)
print(Y)

X shape: torch.Size([2, 3, 3])
K shape: torch.Size([2, 2, 2])
Y shape: torch.Size([2, 2])
tensor([[ 56.,  72.],
        [104., 120.]])


In [6]:
# Multiple Output Channel Cross-Correlation


def corr2d_multi_in_out(
    X: torch.Tensor,  # (in, H, W): (2, 3, 3)
    K: torch.Tensor,  # (out, in, H, W): (3, 2, 2, 2)
) -> torch.Tensor:

    # [out, H, W]
    output_maps: list[torch.Tensor] = [
        # 각 채널당 Convolution 후 누적 가중합 계산
        corr2d_multi_in(
            X,
            K_for_output,
        )
        # for [in, H, W] in [out, in, H, W]
        for K_for_output in K
    ]

    return torch.stack(
        output_maps,
        dim=0,
    )

In [7]:
# Three-Channel Output 검증

# K_multi: [out, in, H, W]
# [3, 2, 2, 2]
K_multi = torch.stack(
    (
        K,
        K + 1,
        K + 2,
    ),
    dim=0,
)

Y_multi = corr2d_multi_in_out(
    X,  # [in, H, W] = [2, 3, 3]
    K_multi,  # [out, in, H, W] = [3, 2, 2, 2]
)

print(
    "X shape:",
    X.shape,
)
print(
    "K_multi shape:",
    K_multi.shape,
)
print(
    "Y_multi shape:",
    Y_multi.shape,
)
print(Y_multi)

X shape: torch.Size([2, 3, 3])
K_multi shape: torch.Size([3, 2, 2, 2])
Y_multi shape: torch.Size([3, 2, 2])
tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])
